This notebook records the cells used for the completed development run. Results are archived in `../reports/development_20260906/`; see `REVIEW.md` there and `../NEXT_CHAT_HANDOFF.md` for current findings. The next task is the missing Phase 4 semantic/localization smoke, not rerunning completed gates.


# Phase 3 full development trajectory

Run the two Python cells below, in order, in the same Kaggle notebook kernel with a GPU enabled and internet available for Git fetch. They use the existing checkout at `/kaggle/working/newpipeline` and the already validated cache at `/kaggle/working/phase2_stage_cache`. If that cache was restored elsewhere, change only `CACHE` in Cell 1 to its actual location. These cells do not extract representations or download a model or dataset.

The local repository was clean on `feat/iclr` at `1a6e0c96681f250659ce703207931289cd9112c2`. Both `scripts/run_phase3_attribute_probes.py` and `src/lger/attribute_probe.py` were inspected, and the runner's actual `--help` was executed successfully. The existing runner supports the requested run; repository code was not modified.

Cell 1 fetches `feat/iclr`, fast-forwards the checkout, verifies HEAD equals the fetched branch, prints the latest runner's help, audits the existing cache, and launches all nine stages with mean pooling, four controls, seeds 0/1/2, 300 epochs, CUDA, and a 256-dimensional Gaussian projection. It keeps the existing learning rate (0.01) and weight decay (0.0001). Each invocation gets a fresh commit-stamped output directory to avoid accepting stale completion files. Git stops on tracked edits or divergent history.

Cell 2 checks the report, configuration, cache identity and record-level official split, exact stage/control/seed combinations, all 26 attribute identities per run, 90 summary rows, and 2,340 per-attribute rows. It verifies metrics, support counts, dimensions, and finite fitted-probe losses, recomputes macro averages, and produces mean/sample-SD summaries, paired deltas, tables, and PNG/PDF trajectory plots. Prevalence runs once per stage with seed `-1`, giving 9 × (3 + 1 + 3 + 3) = 90 runs. No Phase 4 work or official-test evaluation is performed.

The runner's final “STOP: inspect control behavior before launching the full stage trajectory” line is a generic smoke-oriented message printed unconditionally by the current runner. Judge this run's scope from `evaluation_config.json`, `phase3_run_report.json`, and Cell 2's checks.


In [ ]:
# Cell 1: fetch the latest branch, audit the existing cache, and run on Kaggle.
import os, sys, json, hashlib, subprocess
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

PROJECT = Path('/kaggle/working/newpipeline/projects/logit_evidence_routing')
os.chdir(PROJECT)
CACHE = Path('/kaggle/working/phase2_stage_cache')
STAGES = ['vision.early', 'vision.middle', 'vision.late', 'vision.final',
          'projector.output', 'llm.early', 'llm.middle', 'llm.late', 'llm.final']
CONTROLS = ['primary', 'prevalence', 'shuffled_labels', 'random_projection']
SEEDS = [0, 1, 2]
DIGEST = '63cf0e80ec0a24533682467b6f3b23ccded8625ef25d2fb43d012aa0e72179d8'

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_json(path):
    return json.loads(Path(path).read_text())

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def git(*args):
    return subprocess.check_output(['git', *args], text=True).strip()

require(not git('status', '--porcelain', '--untracked-files=no'),
        'Tracked repository changes exist; preserve them before updating.')
subprocess.run(['git', 'fetch', '--no-tags', 'origin',
                '+refs/heads/feat/iclr:refs/remotes/origin/feat/iclr'], check=True)
subprocess.run(['git', 'switch', 'feat/iclr'], check=True)
subprocess.run(['git', 'merge', '--ff-only', 'origin/feat/iclr'], check=True)
HEAD = git('rev-parse', 'HEAD')
require(HEAD == git('rev-parse', 'origin/feat/iclr'), 'HEAD differs from fetched branch.')
print('Running latest fetched feat/iclr:', HEAD, flush=True)
env = dict(os.environ, PYTHONPATH='src', PYTHONUNBUFFERED='1',
           HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
subprocess.run([sys.executable, 'scripts/run_phase3_attribute_probes.py', '--help'],
               env=env, check=True)
import torch
import pandas, numpy, matplotlib  # Fail before training if aggregation dependencies are missing.
require(torch.cuda.is_available(), 'Enable a Kaggle GPU before running this cell.')

def audit_cache():
    # Inspect cached metadata only; never open the official test dataset.
    cfg = read_json(CACHE / 'run_config.json')
    index = read_json(CACHE / 'index.json')
    report = read_json(CACHE / 'validation_report.json')
    digest = hashlib.sha256(json.dumps(cfg, sort_keys=True, separators=(',', ':'),
                                       ensure_ascii=False).encode()).hexdigest()
    require(digest == DIGEST == index['config_digest'] == report['config_digest'],
            'Cache digest differs from the completed Phase 2 gate.')
    require(cfg['purpose'] == 'full_240_image_development_pilot_stage_cache' and
            cfg['official_test_images'] == 0 and index['complete'] is True,
            'Cache is not the completed development cache.')
    expected = {'status': 'PASS', 'images': 240, 'official_test_images': 0,
                'development_split_counts': {'train': 160, 'val': 80},
                'shards': 12, 'reload_validated_records': 240,
                'stage_count_per_record': 9, 'official_test_split_untouched': True}
    for key, value in expected.items():
        require(report.get(key) == value, f'Phase 2 gate mismatch: {key}')
    require(len(index['shards']) == 12, 'Expected 12 shards.')
    images, attribute_ids = [], None
    for shard in index['shards']:
        metadata_path = CACHE / shard['metadata_path']
        require(sha256(metadata_path) == shard['metadata_sha256'], 'Metadata hash changed.')
        require((CACHE / shard['tensor_path']).is_file(), 'Missing cached tensor shard.')
        metadata = read_json(metadata_path)
        require(metadata['complete'] is True and metadata['config_digest'] == DIGEST,
                'Incomplete or incompatible shard metadata.')
        for item in metadata['records']:
            record = item['packed_record']
            image = record['image']
            require(image['official_split'] == 'train', 'Official-test record detected.')
            require(set(record['stages']) == set(STAGES), 'Cached stage set differs.')
            selected = image['selected_attribute_ids']
            if attribute_ids is None:
                attribute_ids = selected
            require(selected == attribute_ids and len(set(selected)) == 26,
                    'Selected attributes changed or count is not 26.')
            images.append(image)
    ids = [int(image['image_id']) for image in images]
    require(len(ids) == len(set(ids)) == 240 and
            ids == cfg['manifest_image_ids'] == [r['image_id'] for r in index['records']],
            'Cached image IDs differ from the frozen manifest.')
    require(Counter(image['development_split'] for image in images) == {'train': 160, 'val': 80},
            'Development split changed.')
    return attribute_ids

ATTRIBUTE_IDS = audit_cache()
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path(f'/kaggle/working/phase3_development_{HEAD[:12]}_{stamp}')
OUT.mkdir(exist_ok=False)
command = [sys.executable, 'scripts/run_phase3_attribute_probes.py',
           '--cache-dir', str(CACHE), '--output-dir', str(OUT),
           '--stages', *STAGES, '--pooling', 'mean', '--controls', *CONTROLS,
           '--seeds', '0', '1', '2', '--epochs', '300',
           '--learning-rate', '0.01', '--weight-decay', '0.0001',
           '--random-projection-dim', '256', '--device', 'cuda']
(OUT / 'invocation.json').write_text(json.dumps(
    {'git_commit': HEAD, 'command': command, 'cwd': str(PROJECT)}, indent=2))
print('Output:', OUT, flush=True)
subprocess.run(command, env=env, check=True)
print('Run completed. Execute Cell 2 to validate and aggregate:', OUT)


In [ ]:
# Cell 2: run after Cell 1 in the same kernel. No probe training occurs here.
os.chdir('/kaggle/working/newpipeline/projects/logit_evidence_routing')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

report = read_json(OUT / 'phase3_run_report.json')
cfg = read_json(OUT / 'evaluation_config.json')
summary = pd.read_csv(OUT / 'attribute_probe_by_stage.csv')
attrs = pd.read_csv(OUT / 'per_attribute_metrics.csv')
require(audit_cache() == ATTRIBUTE_IDS, 'Cache changed after the run.')
expected = {'status': 'PASS', 'cache_config_digest': DIGEST,
            'images': 240, 'train_images': 160, 'validation_images': 80,
            'selected_attributes': 26, 'stages': STAGES, 'pooling': ['mean'],
            'controls': CONTROLS, 'seeds': SEEDS, 'result_rows': 90,
            'per_attribute_rows': 2340, 'official_test_images_used': 0}
for key, value in expected.items():
    require(report.get(key) == value, f'Phase 3 report mismatch: {key}')
expected_cfg = {'git_commit': HEAD, 'stages': STAGES, 'pooling': ['mean'],
                'controls': CONTROLS, 'seeds': SEEDS, 'epochs': 300,
                'device': 'cuda', 'random_projection_dim': 256,
                'learning_rate': 0.01, 'weight_decay': 0.0001,
                'normalization': 'training_feature_mean_std_v1',
                'threshold_policy': 'training_f1_only_v1',
                'official_test_images_used': 0,
                'cache_dir': str(CACHE.resolve()),
                'cache_validation_sha256': sha256(CACHE / 'validation_report.json')}
for key, value in expected_cfg.items():
    require(cfg.get(key) == value, f'Evaluation config mismatch: {key}')
require(len(summary) == 90 and len(attrs) == 2340, 'Wrong CSV row counts.')
KEY = ['stage', 'pooling', 'control', 'seed']
expected_keys = {(s, 'mean', c, seed) for s in STAGES for c in CONTROLS
                 for seed in ([-1] if c == 'prevalence' else SEEDS)}
require(not summary.duplicated(KEY).any() and
        set(summary[KEY].itertuples(index=False, name=None)) == expected_keys,
        'Missing, duplicate, or unexpected stage/control/seed rows.')
require(not attrs.duplicated(KEY + ['attribute_id']).any() and
        set(attrs[KEY].itertuples(index=False, name=None)) == expected_keys,
        'Per-attribute run keys differ.')
for key, group in attrs.groupby(KEY):
    require(len(group) == 26 and set(group.attribute_id) == set(ATTRIBUTE_IDS),
            f'Attribute identities differ for {key}')
for column, value in [('train_examples', 160), ('validation_examples', 80),
                      ('selected_attributes', 26), ('evaluable_auroc_attributes', 26),
                      ('evaluable_f1_attributes', 26)]:
    require(summary[column].eq(value).all(), f'Unexpected {column}')
for table, columns in [(summary, ['macro_auroc', 'macro_f1']),
                       (attrs, ['auroc', 'f1', 'threshold_selected_on_train',
                                'train_prevalence'])]:
    require(np.isfinite(table[columns].to_numpy(dtype=float)).all(),
            f'Nonfinite values in {columns}; inspect support before interpreting results.')
for table, columns in [(summary, ['macro_auroc', 'macro_f1']), (attrs, ['auroc', 'f1'])]:
    require(table[columns].ge(0).all().all() and table[columns].le(1).all().all(),
            'Metric outside [0, 1].')
trained = attrs.control.ne('prevalence')
require(np.isfinite(attrs.loc[trained, 'train_loss']).all() and
        attrs.loc[trained, 'train_loss'].ge(0).all(), 'Nonfinite or negative fitted-probe loss.')
require(attrs.loc[~trained, 'train_loss'].isna().all(),
        'Prevalence loss should be NaN: no probe is fitted.')
require(attrs.groupby(KEY).train_loss.nunique().le(1).all(),
        'A shared probe loss should be constant across its attribute rows.')
for split, maximum in [('train', 160), ('validation', 80)]:
    require(attrs[f'{split}_observed'].eq(
        attrs[f'{split}_positive'] + attrs[f'{split}_negative']).all(), 'Support counts differ.')
    require(attrs[f'{split}_observed'].between(1, maximum).all(), 'Invalid observed count.')
    require(attrs[f'{split}_positive'].gt(0).all() and attrs[f'{split}_negative'].gt(0).all(),
            'An attribute lacks positive or negative support.')
expected_dims = summary.apply(lambda r: 256 if r.control == 'random_projection'
                              else (1024 if r.stage.startswith('vision.') else 4096), axis=1)
require(summary.feature_dim.eq(expected_dims).all(), 'Feature dimensions differ.')
recomputed = attrs.groupby(KEY)[['auroc', 'f1']].mean()
indexed = summary.set_index(KEY).loc[recomputed.index]
require(np.allclose(indexed[['macro_auroc', 'macro_f1']], recomputed,
                    rtol=1e-7, atol=1e-9), 'Macro metrics disagree with attribute rows.')
require(np.allclose(summary.loc[summary.control.eq('prevalence'), 'macro_auroc'], 0.5),
        'Constant-score prevalence AUROC should be 0.5.')

# Sample SD (ddof=1); prevalence has n=1, so its seed SD is undefined.
aggregated = summary.groupby(['stage', 'control'], sort=False).agg(
    n=('seed', 'size'), auroc_mean=('macro_auroc', 'mean'),
    auroc_sd=('macro_auroc', 'std'), f1_mean=('macro_f1', 'mean'),
    f1_sd=('macro_f1', 'std')).reset_index()
loss_by_run = attrs.loc[trained].groupby(KEY).train_loss.first()
loss_stats = loss_by_run.groupby(['stage', 'control']).agg(['mean', 'std'])
loss_stats = loss_stats.rename(columns={'mean': 'train_loss_mean', 'std': 'train_loss_sd'})
aggregated = aggregated.merge(loss_stats, on=['stage', 'control'], how='left')

# Pair by seed before summarizing differences; never subtract SDs.
primary = summary[summary.control.eq('primary')]
delta_frames = []
for control in ['shuffled_labels', 'random_projection']:
    pairs = primary.merge(summary[summary.control.eq(control)],
                          on=['stage', 'pooling', 'seed'], suffixes=('_p', '_c'),
                          validate='one_to_one')
    delta = pairs[['stage', 'pooling', 'seed']].copy()
    delta['comparison'] = 'primary-minus-' + control
    for metric in ['auroc', 'f1']:
        delta[metric + '_delta'] = pairs['macro_' + metric + '_p'] - pairs['macro_' + metric + '_c']
    delta_frames.append(delta)
deltas = pd.concat(delta_frames, ignore_index=True)
delta_stats = deltas.groupby(['stage', 'comparison'], sort=False).agg(
    n=('seed', 'size'), auroc_delta_mean=('auroc_delta', 'mean'),
    auroc_delta_sd=('auroc_delta', 'std'), f1_delta_mean=('f1_delta', 'mean'),
    f1_delta_sd=('f1_delta', 'std')).reset_index()

def mean_sd(mean, sd):
    return f'{mean:.4f} ± {sd:.4f}' if pd.notna(sd) else f'{mean:.4f} (n=1)'

for metric in ['auroc', 'f1']:
    readable = pd.DataFrame(index=pd.Index(STAGES, name='stage'))
    for control in CONTROLS:
        rows = aggregated[aggregated.control.eq(control)].set_index('stage').loc[STAGES]
        readable[control] = [mean_sd(m, s) for m, s in
                             zip(rows[metric + '_mean'], rows[metric + '_sd'])]
    for control in ['shuffled_labels', 'random_projection']:
        rows = delta_stats[delta_stats.comparison.eq('primary-minus-' + control)].set_index('stage').loc[STAGES]
        readable['Δ primary − ' + control] = [mean_sd(m, s) for m, s in
            zip(rows[metric + '_delta_mean'], rows[metric + '_delta_sd'])]
    print(f'\nMacro {metric.upper()}: mean ± sample SD across seeds (0, 1, 2)')
    display(readable)
    readable.to_csv(OUT / f'{metric}_trajectory_readable.csv')

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
x = np.arange(len(STAGES))
for column, metric in enumerate(['auroc', 'f1']):
    for control in CONTROLS:
        rows = aggregated[aggregated.control.eq(control)].set_index('stage').loc[STAGES]
        axes[0, column].errorbar(x, rows[metric + '_mean'],
            yerr=None if control == 'prevalence' else rows[metric + '_sd'],
            marker='o', capsize=3, label=control)
    axes[0, column].set(title=f'Macro {metric.upper()}: mean ± seed SD', ylim=(-0.03, 1.03))
    for comparison, rows in delta_stats.groupby('comparison', sort=False):
        rows = rows.set_index('stage').loc[STAGES]
        axes[1, column].errorbar(x, rows[metric + '_delta_mean'],
            yerr=rows[metric + '_delta_sd'], marker='o', capsize=3, label=comparison)
    axes[1, column].axhline(0, color='black', linewidth=0.8)
    axes[1, column].set_title(f'Paired {metric.upper()} differences: mean ± seed SD')
for ax in axes.flat:
    ax.set_xticks(x, STAGES, rotation=45, ha='right')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.suptitle('Phase 3 development only · 160 train / 80 validation · 26 attributes · mean pooling')
fig.tight_layout()
fig.savefig(OUT / 'stage_trajectory.png', dpi=180, bbox_inches='tight')
fig.savefig(OUT / 'stage_trajectory.pdf', bbox_inches='tight')
plt.show()
aggregated.to_csv(OUT / 'stage_seed_summary.csv', index=False)
deltas.to_csv(OUT / 'paired_seed_deltas.csv', index=False)
delta_stats.to_csv(OUT / 'paired_delta_summary.csv', index=False)
(OUT / 'phase3_aggregation_report.json').write_text(json.dumps({
    'status': 'PASS', 'git_commit': HEAD, 'cache_config_digest': DIGEST,
    'result_rows': len(summary), 'per_attribute_rows': len(attrs),
    'official_test_images_used': 0, 'official_test_split_untouched': True,
    'sd_definition': 'sample SD across seeds, ddof=1; not a confidence interval',
    'prevalence_train_loss': 'NaN by design; no fitting'}, indent=2))
print('\nValidation/aggregation PASS:', OUT)
print('Finite fitted losses:', float(loss_by_run.min()), 'to', float(loss_by_run.max()))
print('Prevalence training loss is intentionally NaN. Official test remains untouched.')


## Reading the results

- **Primary vs shuffled labels:** the shuffled control permutes each attribute's observed training targets, preserving its observed/missing mask and class counts. It is evaluated against the true validation targets. A consistent positive primary-minus-shuffled AUROC gap supports recoverable attribute information. Shuffled AUROC should be broadly near 0.5, with finite-sample variation. Suspiciously strong shuffled results warrant investigation; passing this control does not prove that all leakage or confounding is absent.
- **Prevalence:** each attribute receives its development-training prevalence as a constant score. AUROC should be 0.5 when both validation classes are present. F1 is zero when all relevant prevalences are below the fixed 0.5 probability threshold, as in the smoke; zero F1 is not a universal requirement for this baseline. No probe is fitted, so its training loss is intentionally NaN and its seed SD is undefined (n=1). The validator permits those specific cases.
- **Random projection is a dimensionality control, not a null baseline.** A frozen Gaussian map compresses the original features to 256 dimensions, then a supervised probe is trained on the real labels. Signal is retained. Comparable performance suggests that much of the linearly recoverable information survives compression under this protocol. A negative primary-minus-projection delta can reflect regularization, optimization, or sampling effects; it does not establish that projection creates information or is generally superior. Even equal output dimensions do not equalize every property of the representations.
- **Trajectory:** compare primary AUROC and F1 across all nine stages alongside both paired control gaps. Mean pooling measures global attribute recoverability; these probes alone do not establish spatial localization, causal use of attributes by the VLM, or a causal information bottleneck. Small stage differences deserve restraint. F1 also depends on training-selected thresholds; prevalence uses a fixed threshold, so its F1 is not threshold-matched to the trained probes.
- **Uncertainty:** SD is sample SD (ddof=1) across the three seeds, not a confidence interval. Delta SD is computed from the three within-seed differences, not by subtracting SDs. The same 80 validation images are reused; seed variation does not measure image-sampling uncertainty or provide an independent significance test. Random-projection variation includes both the projection and probe initialization.
- **Validation limits:** the existing Phase 2 reload-validation PASS is trusted for tensor integrity; these cells recheck metadata hashes and metadata identities without repeating the full 7.3 GB tensor hash/reload gate. Every validation AUROC must be finite with both classes supported; if that fails, inspect the offending attributes and report the issue without silently dropping attributes or changing metrics. A finite recorded loss checks the final logged training scalar, not convergence or every epoch's loss history.

Review this full development result before proposing any next phase. Keep the official CUB test split untouched until attributes, layers, prompts, parsers, and metrics are frozen.

## Saved Kaggle outputs

The unique directory printed as `OUT` contains the runner's raw tables and reports plus:

- `stage_seed_summary.csv`: mean and sample SD for AUROC, F1, and fitted training loss.
- `paired_seed_deltas.csv` and `paired_delta_summary.csv`: paired differences for both metrics.
- `auroc_trajectory_readable.csv` and `f1_trajectory_readable.csv`: ordered stage tables.
- `stage_trajectory.png` and `stage_trajectory.pdf`: control trajectories and paired deltas.
- `phase3_aggregation_report.json`: successful validation/aggregation gate.

Local checks: both cells passed Python syntax checks; the real runner's `--help` was executed. Aggregation was exercised on tiny synthetic CSV/JSON fixtures, including rejection of a missing result row, an infinite fitted loss, and an official-test metadata record. Plotting was syntax-checked but not rendered locally because the available local runtime lacks matplotlib. No actual VLM extraction, probe fitting, model download, or dataset download was performed locally. The full run and plot rendering remain for Kaggle.
